# ⚽ Joueurs et effectif — Ligue des champions

Cette analyse produit `output/joueurs_par_match.csv`, avec une ligne par équipe et par match de Ligue des champions.

Elle calcule la valeur marchande des joueurs ayant joué, l'âge moyen des joueurs alignés, le nombre de joueurs distincts utilisés par club et saison en C1 et en championnat national, ainsi que la part des buts du meilleur buteur dans les buts du club. Elle compare ensuite ces indicateurs au résultat du match.

**Pourquoi la valeur avant le match ?** Une valeur marchande publiée après le match ne pouvait pas être connue au moment du match. Le calcul retient donc pour chaque joueur la dernière valorisation strictement antérieure à la date du match. Cette précaution évite d'introduire des informations futures dans l'indicateur.

`player_valuations.csv` est déjà présent dans `data/`. S'il manque sur une autre installation, il faut le télécharger depuis le jeu Kaggle *Player Scores* de Transfermarkt et le placer dans `data/` avant l'exécution.

Les apparitions historiques peuvent être incomplètes. Le CSV conserve les nombres de joueurs et la couverture des valorisations pour rendre cette limite visible. Les comparaisons de valeur et d'âge exigent au moins 11 joueurs observés par équipe et une couverture des données d'au moins 90 %. Une association entre indicateur et résultat ne démontre pas une causalité.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Le notebook fonctionne depuis la racine du projet ou depuis analyse/.
DOSSIER_PROJET = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()),
    None,
)
if DOSSIER_PROJET is None:
    raise FileNotFoundError("Impossible de trouver le dossier data/ depuis le répertoire courant.")
DOSSIER_DATA = DOSSIER_PROJET / "data"
DOSSIER_SORTIE = DOSSIER_PROJET / "output"
FICHIER_VALORISATIONS = DOSSIER_DATA / "player_valuations.csv"
if not FICHIER_VALORISATIONS.is_file():
    raise FileNotFoundError(
        "player_valuations.csv est absent de data/. Téléchargez-le depuis Kaggle "
        "(Player Scores de Transfermarkt) et relancez le notebook."
    )

# Charger les seules colonnes nécessaires, avec dates et identifiants au bon format.
matchs = pd.read_csv(
    DOSSIER_DATA / "games.csv",
    usecols=[
        "game_id", "competition_id", "season", "date", "home_club_id", "away_club_id",
        "home_club_name", "away_club_name", "home_club_goals", "away_club_goals",
    ],
    parse_dates=["date"],
)
apparitions_source = pd.read_csv(
    DOSSIER_DATA / "appearances.csv",
    usecols=["game_id", "player_id", "player_club_id", "minutes_played", "goals"],
)
competitions = pd.read_csv(
    DOSSIER_DATA / "competitions.csv",
    usecols=["competition_id", "type"],
)
players = pd.read_csv(
    DOSSIER_DATA / "players.csv",
    usecols=["player_id", "date_of_birth"],
    parse_dates=["date_of_birth"],
)

# Joindre les dates et saisons officielles des matchs aux apparitions.
matchs_pour_jonction = matchs[["game_id", "competition_id", "season", "date"]]
apparitions = apparitions_source.merge(
    matchs_pour_jonction,
    on="game_id",
    how="inner",
    validate="many_to_one",
)
apparitions = apparitions.loc[apparitions["minutes_played"].gt(0)].copy()
types_competitions = competitions.set_index("competition_id")["type"]
apparitions["type_competition"] = apparitions["competition_id"].map(types_competitions)

print(f"Matchs chargés : {len(matchs):,}")
print(f"Apparitions avec minutes jouées et match correspondant : {len(apparitions):,}")
print(f"Fichier des valorisations présent : {FICHIER_VALORISATIONS.name}")

In [ ]:
# Rotation et dépendance au meilleur buteur par club, compétition et saison.
# On ne compte comme utilisé qu'un joueur ayant effectivement joué.
competitions_a_etudier = set(
    competitions.loc[competitions["type"].eq("domestic_league"), "competition_id"]
) | {"CL"}
apparitions_competitions = apparitions.loc[
    apparitions["competition_id"].isin(competitions_a_etudier)
].copy()
cles_club_saison_competition = ["player_club_id", "competition_id", "season"]

rotation = (
    apparitions_competitions.groupby(cles_club_saison_competition)["player_id"]
    .nunique()
    .rename("nb_joueurs_utilises")
    .reset_index()
)
buts_par_joueur = (
    apparitions_competitions.groupby(cles_club_saison_competition + ["player_id"])["goals"]
    .sum()
    .reset_index()
)
buts_club = (
    buts_par_joueur.groupby(cles_club_saison_competition)["goals"]
    .sum()
    .rename("buts_club_saison")
    .reset_index()
)
meilleur_buteur = (
    buts_par_joueur.groupby(cles_club_saison_competition)["goals"]
    .max()
    .rename("buts_meilleur_buteur")
    .reset_index()
)
indicateurs_saison = (
    rotation.merge(buts_club, on=cles_club_saison_competition, validate="one_to_one")
    .merge(meilleur_buteur, on=cles_club_saison_competition, validate="one_to_one")
)
indicateurs_saison["part_buts_meilleur_buteur"] = (
    indicateurs_saison["buts_meilleur_buteur"]
    / indicateurs_saison["buts_club_saison"].where(indicateurs_saison["buts_club_saison"].gt(0))
)

# Identifier le championnat national effectivement observé pour chaque club-saison.
# Le club peut changer de championnat au cours de l'historique : on utilise ici
# la compétition nationale comptant le plus d'apparitions pour ce club cette saison.
apparitions_championnat = apparitions_competitions.loc[
    apparitions_competitions["type_competition"].eq("domestic_league")
]
championnat_club_saison = (
    apparitions_championnat.groupby(["player_club_id", "season", "competition_id"])
    .size()
    .rename("nb_apparitions_championnat")
    .reset_index()
    .sort_values(
        ["player_club_id", "season", "nb_apparitions_championnat", "competition_id"],
        ascending=[True, True, False, True],
    )
    .drop_duplicates(["player_club_id", "season"])
    .rename(columns={"competition_id": "competition_championnat"})
)
indicateurs_saison_cl = indicateurs_saison.loc[
    indicateurs_saison["competition_id"].eq("CL"),
    [
        "player_club_id", "season", "nb_joueurs_utilises",
        "buts_club_saison", "buts_meilleur_buteur", "part_buts_meilleur_buteur",
    ],
].rename(
    columns={
        "nb_joueurs_utilises": "nb_joueurs_utilises_cl",
        "buts_club_saison": "buts_club_cl",
        "buts_meilleur_buteur": "buts_meilleur_buteur_cl",
        "part_buts_meilleur_buteur": "part_buts_meilleur_buteur_cl",
    }
)
indicateurs_saison_cl = indicateurs_saison_cl.merge(
    championnat_club_saison,
    on=["player_club_id", "season"],
    how="left",
    validate="one_to_one",
)
rotation_championnat = rotation.rename(
    columns={
        "competition_id": "competition_championnat",
        "nb_joueurs_utilises": "nb_joueurs_utilises_championnat",
    }
)[["player_club_id", "competition_championnat", "season", "nb_joueurs_utilises_championnat"]]
indicateurs_saison_cl = indicateurs_saison_cl.merge(
    rotation_championnat,
    on=["player_club_id", "competition_championnat", "season"],
    how="left",
    validate="many_to_one",
)

print(f"Compétitions club-saison calculées : {len(indicateurs_saison):,}")
print(f"Saisons-club de C1 avec indicateurs : {len(indicateurs_saison_cl):,}")
print("Rotation moyenne par type de compétition :")
display(
    indicateurs_saison.merge(competitions, on="competition_id", how="left")
    .groupby("type")["nb_joueurs_utilises"]
    .agg(nb_club_saisons="size", moyenne="mean", mediane="median")
    .round(1)
    .reset_index()
)

In [ ]:
# Valeur pré-match et âge moyen des joueurs, au niveau match-club de C1.
apparitions_cl = apparitions.loc[apparitions["competition_id"].eq("CL")].copy()
valorisations = pd.read_csv(
    FICHIER_VALORISATIONS,
    usecols=["player_id", "date", "market_value_in_eur"],
    parse_dates=["date"],
).dropna(subset=["date"])
# Une seule observation par joueur et date pour un rattachement as-of déterministe.
valorisations = (
    valorisations.sort_values(["date", "player_id"])
    .drop_duplicates(["player_id", "date"], keep="last")
    .sort_values("date")
)
apparitions_cl = apparitions_cl.sort_values("date")
apparitions_valorisees = pd.merge_asof(
    apparitions_cl,
    valorisations,
    on="date",
    by="player_id",
    direction="backward",
    allow_exact_matches=False,
)
valeur_match = (
    apparitions_valorisees.groupby(["game_id", "player_club_id"])
    .agg(
        valeur_effectif_eur_avant_match=(
            "market_value_in_eur", lambda valeurs: valeurs.sum(min_count=1)
        ),
        nb_joueurs_alignes=("player_id", "nunique"),
        nb_joueurs_avec_valeur=("market_value_in_eur", "count"),
    )
    .reset_index()
)
valeur_match["couverture_valeur"] = (
    valeur_match["nb_joueurs_avec_valeur"] / valeur_match["nb_joueurs_alignes"]
)

apparitions_age = apparitions_cl.merge(
    players,
    on="player_id",
    how="left",
    validate="many_to_one",
)
apparitions_age["age_au_match"] = (
    (apparitions_age["date"] - apparitions_age["date_of_birth"]).dt.days / 365.2425
)
age_match = (
    apparitions_age.groupby(["game_id", "player_club_id"])
    .agg(
        age_moyen_joueurs_alignes=("age_au_match", "mean"),
        nb_joueurs_avec_date_naissance=("age_au_match", "count"),
    )
    .reset_index()
)
age_match["couverture_date_naissance"] = (
    age_match["nb_joueurs_avec_date_naissance"]
    / valeur_match.set_index(["game_id", "player_club_id"])["nb_joueurs_alignes"].reindex(
        pd.MultiIndex.from_frame(age_match[["game_id", "player_club_id"]])
    ).to_numpy()
)
indicateurs_match = valeur_match.merge(
    age_match,
    on=["game_id", "player_club_id"],
    how="outer",
    validate="one_to_one",
)

# Construire une ligne par club et match, depuis le score officiel du match.
matchs_cl = matchs.loc[
    matchs["competition_id"].eq("CL")
    & matchs["home_club_goals"].notna()
    & matchs["away_club_goals"].notna()
].copy()
colonnes_match = ["game_id", "date", "season", "home_club_id", "away_club_id", "home_club_name", "away_club_name", "home_club_goals", "away_club_goals"]
domicile = matchs_cl[colonnes_match].rename(
    columns={
        "home_club_id": "club_id", "away_club_id": "adversaire_id",
        "home_club_name": "club", "away_club_name": "adversaire",
        "home_club_goals": "buts_pour", "away_club_goals": "buts_contre",
    }
)
domicile["lieu"] = "Domicile"
exterieur = matchs_cl[colonnes_match].rename(
    columns={
        "away_club_id": "club_id", "home_club_id": "adversaire_id",
        "away_club_name": "club", "home_club_name": "adversaire",
        "away_club_goals": "buts_pour", "home_club_goals": "buts_contre",
    }
)
exterieur["lieu"] = "Extérieur"
equipes_match = pd.concat([domicile, exterieur], ignore_index=True)
equipes_match = equipes_match.merge(
    indicateurs_match,
    left_on=["game_id", "club_id"],
    right_on=["game_id", "player_club_id"],
    how="left",
    validate="one_to_one",
).drop(columns="player_club_id")
equipes_match = equipes_match.merge(
    indicateurs_saison_cl,
    left_on=["club_id", "season"],
    right_on=["player_club_id", "season"],
    how="left",
    validate="many_to_one",
).drop(columns="player_club_id")
equipes_match["resultat"] = np.select(
    [
        equipes_match["buts_pour"].gt(equipes_match["buts_contre"]),
        equipes_match["buts_pour"].eq(equipes_match["buts_contre"]),
    ],
    ["Victoire", "Nul"],
    default="Défaite",
)

print(f"Matchs de C1 avec score : {len(matchs_cl):,}")
print(f"Lignes club-match produites : {len(equipes_match):,}")
print(f"Matchs avec au moins 11 joueurs observés dans les deux équipes : "
      f"{(equipes_match.groupby('game_id')['nb_joueurs_alignes'].min().ge(11).sum()):,}")
display(
    equipes_match[
        [
            "game_id", "date", "season", "club", "adversaire", "lieu", "resultat",
            "valeur_effectif_eur_avant_match", "couverture_valeur",
            "age_moyen_joueurs_alignes", "nb_joueurs_utilises_championnat",
            "nb_joueurs_utilises_cl", "part_buts_meilleur_buteur_cl",
        ]
    ].head(10)
)

In [ ]:
# Comparer les indicateurs à l'issue des matchs de C1.
# « Plus favorisée » signifie ici une valeur strictement plus élevée ; les égalités
# sont comptabilisées séparément. Résultats descriptifs, pas une relation causale.
indicateurs_a_comparer = {
    "valeur_effectif_eur_avant_match": ("Valeur de l'effectif avant le match (€)", "qualite_match"),
    "age_moyen_joueurs_alignes": ("Âge moyen des joueurs alignés", "qualite_match"),
    "nb_joueurs_utilises_championnat": ("Rotation — championnat national", "saison"),
    "nb_joueurs_utilises_cl": ("Rotation — Ligue des champions", "saison"),
    "part_buts_meilleur_buteur_cl": ("Dépendance au meilleur buteur — C1", "saison"),
}
domicile_index = equipes_match.loc[equipes_match["lieu"].eq("Domicile")].set_index("game_id")
exterieur_index = equipes_match.loc[equipes_match["lieu"].eq("Extérieur")].set_index("game_id")
resume = []

for colonne, (libelle, type_indicateur) in indicateurs_a_comparer.items():
    colonnes = [colonne, "buts_pour", "buts_contre"]
    if type_indicateur == "qualite_match":
        colonnes += ["nb_joueurs_alignes", "couverture_valeur", "couverture_date_naissance"]
    paires = domicile_index[colonnes].join(
        exterieur_index[colonnes], lsuffix="_domicile", rsuffix="_exterieur", how="inner"
    )
    difference = paires[f"{colonne}_domicile"] - paires[f"{colonne}_exterieur"]
    eligible = difference.notna()
    if type_indicateur == "qualite_match":
        eligible &= (
            paires["nb_joueurs_alignes_domicile"].ge(11)
            & paires["nb_joueurs_alignes_exterieur"].ge(11)
        )
        if colonne == "valeur_effectif_eur_avant_match":
            eligible &= (
                paires["couverture_valeur_domicile"].ge(0.9)
                & paires["couverture_valeur_exterieur"].ge(0.9)
            )
        else:
            eligible &= (
                paires["couverture_date_naissance_domicile"].ge(0.9)
                & paires["couverture_date_naissance_exterieur"].ge(0.9)
            )
    differentes = eligible & difference.ne(0)
    egales = eligible & difference.eq(0)
    valeur_plus_haute_a_domicile = difference.loc[differentes].gt(0)
    buts_cote_plus_haut = np.where(
        valeur_plus_haute_a_domicile,
        paires.loc[differentes, "buts_pour_domicile"],
        paires.loc[differentes, "buts_pour_exterieur"],
    )
    buts_cote_plus_bas = np.where(
        valeur_plus_haute_a_domicile,
        paires.loc[differentes, "buts_contre_domicile"],
        paires.loc[differentes, "buts_contre_exterieur"],
    )
    victoires = int((buts_cote_plus_haut > buts_cote_plus_bas).sum())
    nuls = int((buts_cote_plus_haut == buts_cote_plus_bas).sum())
    defaites = int((buts_cote_plus_haut < buts_cote_plus_bas).sum())
    resume.append(
        {
            "Indicateur": libelle,
            "Matchs éligibles": int(eligible.sum()),
            "Valeurs différentes": int(differentes.sum()),
            "Valeurs égales": int(egales.sum()),
            "Victoires de l'équipe à la valeur la plus élevée": victoires,
            "Nuls": nuls,
            "Défaites": defaites,
            "Taux de victoire hors nuls (%)": round(100 * victoires / (victoires + defaites), 1)
            if victoires + defaites
            else np.nan,
        }
    )

comparaison_indicateurs = pd.DataFrame(resume)
display(comparaison_indicateurs)

# Livrable demandé : une ligne par club et match de C1.
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
colonnes_livrable = [
    "game_id", "date", "season", "club_id", "club", "adversaire_id", "adversaire",
    "lieu", "buts_pour", "buts_contre", "resultat",
    "valeur_effectif_eur_avant_match", "nb_joueurs_alignes", "nb_joueurs_avec_valeur",
    "couverture_valeur", "age_moyen_joueurs_alignes", "nb_joueurs_avec_date_naissance",
    "couverture_date_naissance", "competition_championnat",
    "nb_joueurs_utilises_championnat", "nb_joueurs_utilises_cl",
    "buts_meilleur_buteur_cl", "buts_club_cl", "part_buts_meilleur_buteur_cl",
]
livrable = equipes_match[colonnes_livrable].sort_values(["date", "game_id", "lieu"])
chemin_livrable = DOSSIER_SORTIE / "joueurs_par_match.csv"
livrable.to_csv(chemin_livrable, index=False)
print(f"Livrable créé : {chemin_livrable}")
print(f"Dimensions : {livrable.shape[0]:,} lignes × {livrable.shape[1]} colonnes")